In [1]:
# Install dependencies
!pip install nltk spacy pandas
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 94.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# Import modules & download NLTK data
import nltk
from nltk.tokenize import TweetTokenizer, word_tokenize
from nltk.corpus import stopwords
import pandas as pd
import spacy
from spacy import displacy

# Download required NLTK taggers and chunk models
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('universal_tagset')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Unzipping taggers/universal_tagset.zip.


True

In [3]:
# Tokenizing and Tagging Twitter Data
sample_tweets = [
    "Loving the new #PyTorch 2.0 release! @OpenAI models run so fast rn 🔥🚀",
    "Can't believe the flight got delayed again... smh @Delta #traveldiaries",
    "NLP is amazing! Check out https://huggingface.co for open-source models."
]

# Initialize TweetTokenizer: preserves handles, URLs, and reduces repeated characters
tweet_tokenizer = TweetTokenizer(preserve_case=False, strip_handles=False, reduce_len=True)

tweet_tag_records = []

for idx, tweet in enumerate(sample_tweets, start=1):
    # 1. Tokenize using Twitter-aware tokenizer
    tokens = tweet_tokenizer.tokenize(tweet)

    # 2. Tag with Penn Treebank tagset & Universal tagset
    tags_ptb = nltk.pos_tag(tokens)
    tags_univ = nltk.pos_tag(tokens, tagset='universal')

    for (w, ptb), (_, univ) in zip(tags_ptb, tags_univ):
        tweet_tag_records.append({
            "Tweet_ID": f"Tweet {idx}",
            "Token": w,
            "Penn_Treebank_Tag": ptb,
            "Universal_Tag": univ
        })

df_twitter_tags = pd.DataFrame(tweet_tag_records)
print("--- Twitter POS Tagging Sample (First 15 Tokens) ---")
print(df_twitter_tags.head(15).to_string(index=False))

--- Twitter POS Tagging Sample (First 15 Tokens) ---
Tweet_ID    Token Penn_Treebank_Tag Universal_Tag
 Tweet 1   loving               VBG          VERB
 Tweet 1      the                DT           DET
 Tweet 1      new                JJ           ADJ
 Tweet 1 #pytorch                NN          NOUN
 Tweet 1      2.0                CD           NUM
 Tweet 1  release                NN          NOUN
 Tweet 1        !                 .             .
 Tweet 1  @openai                JJ           ADJ
 Tweet 1   models               NNS          NOUN
 Tweet 1      run               VBP          VERB
 Tweet 1       so                RB           ADV
 Tweet 1     fast                RB           ADV
 Tweet 1       rn                JJ           ADJ
 Tweet 1        🔥               NNP          NOUN
 Tweet 1        🚀                NN          NOUN


In [4]:
# Distribution of Twitter Universal POS Tags
tag_distribution = df_twitter_tags['Universal_Tag'].value_counts()
print("\n--- Universal POS Tag Distribution across Sample Tweets ---")
print(tag_distribution)


--- Universal POS Tag Distribution across Sample Tweets ---
Universal_Tag
NOUN    13
VERB     7
ADJ      5
.        4
ADV      3
DET      2
NUM      1
PRT      1
ADP      1
Name: count, dtype: int64


In [5]:
# Noun Phrase (NP) and Verb Phrase (VP) Chunking
sentence = "The clever machine learning engineer quickly solved the complex system error."

# 1. Word Tokenization & POS Tagging
sentence_tokens = word_tokenize(sentence)
tagged_sentence = nltk.pos_tag(sentence_tokens)

print("--- POS-Tagged Sentence ---")
print(tagged_sentence)

# 2. Define a Regular Expression Chunk Grammar
# NP: Optional determiner, optional adverbs/adjectives, one or more nouns
# VP: Optional adverb, followed by a verb, followed optionally by a particle or preposition
chunk_grammar = r"""
    NP: {<DT>?<JJ.*|RB>*<NN.*>+}
    VP: {<RB.*>*<VB.*>}
"""

# 3. Create Chunk Parser
chunk_parser = nltk.RegexpParser(chunk_grammar)
chunk_tree = chunk_parser.parse(tagged_sentence)

print("\n--- Generated Chunk Tree Structure ---")
print(chunk_tree)

# 4. Extract Parsed Chunks and IOB Tags
from nltk.chunk import tree2conlltags
iob_tags = tree2conlltags(chunk_tree)

df_chunks = pd.DataFrame(iob_tags, columns=["Token", "POS_Tag", "IOB_Tag"])
print("\n--- IOB Chunk Representations ---")
print(df_chunks.to_string(index=False))

--- POS-Tagged Sentence ---
[('The', 'DT'), ('clever', 'NN'), ('machine', 'NN'), ('learning', 'VBG'), ('engineer', 'JJ'), ('quickly', 'RB'), ('solved', 'VBD'), ('the', 'DT'), ('complex', 'JJ'), ('system', 'NN'), ('error', 'NN'), ('.', '.')]

--- Generated Chunk Tree Structure ---
(S
  (NP The/DT clever/NN machine/NN)
  (VP learning/VBG)
  engineer/JJ
  (VP quickly/RB solved/VBD)
  (NP the/DT complex/JJ system/NN error/NN)
  ./.)

--- IOB Chunk Representations ---
   Token POS_Tag IOB_Tag
     The      DT    B-NP
  clever      NN    I-NP
 machine      NN    I-NP
learning     VBG    B-VP
engineer      JJ       O
 quickly      RB    B-VP
  solved     VBD    I-VP
     the      DT    B-NP
 complex      JJ    I-NP
  system      NN    I-NP
   error      NN    I-NP
       .       .       O


In [6]:
# Chinking Pattern Example
# Chunk the entire sentence first, then chink (carve out) prepositions and verbs
chink_grammar = r"""
    NP_Chunk:
        {<.*>+}          # Chunk everything
        }<IN|VB.*|CC>+{  # Chink (remove) prepositions, verbs, and conjunctions
"""
chink_parser = nltk.RegexpParser(chink_grammar)
chink_tree = chink_parser.parse(tagged_sentence)

print("--- Tree after Chinking ---")
print(chink_tree)

--- Tree after Chinking ---
(S
  (NP_Chunk The/DT clever/NN machine/NN)
  learning/VBG
  (NP_Chunk engineer/JJ quickly/RB)
  solved/VBD
  (NP_Chunk the/DT complex/JJ system/NN error/NN ./.))


In [7]:
# spaCy POS Tagging and Dependency Parsing
nlp = spacy.load("en_core_web_sm")

doc = nlp("The clever machine learning engineer quickly solved the complex system error.")

parsed_data = []
for token in doc:
    parsed_data.append({
        "Token": token.text,
        "Lemma": token.lemma_,
        "POS (Fine)": token.tag_,
        "POS (Coarse)": token.pos_,
        "Dependency": token.dep_,
        "Head": token.head.text
    })

df_spacy_parsed = pd.DataFrame(parsed_data)
print("--- spaCy Detailed Syntactic & Dependency Analysis ---")
print(df_spacy_parsed.to_string(index=False))

--- spaCy Detailed Syntactic & Dependency Analysis ---
   Token    Lemma POS (Fine) POS (Coarse) Dependency     Head
     The      the         DT          DET        det  machine
  clever   clever         JJ          ADJ       amod  machine
 machine  machine         NN         NOUN   compound engineer
learning    learn        VBG         VERB        acl  machine
engineer engineer         NN         NOUN      nsubj   solved
 quickly  quickly         RB          ADV     advmod   solved
  solved    solve        VBD         VERB       ROOT   solved
     the      the         DT          DET        det    error
 complex  complex         JJ          ADJ       amod    error
  system   system         NN         NOUN   compound    error
   error    error         NN         NOUN       dobj   solved
       .        .          .        PUNCT      punct   solved


In [8]:
# Extracting Noun Chunks with spaCy
print("\n--- Built-in spaCy Noun Chunks ---")
for chunk in doc.noun_chunks:
    print(f"Phrase: {chunk.text:<35} | Root: {chunk.root.text:<10} | Dependency: {chunk.root.dep_}")

# Visualize dependency graph (in a Jupyter environment)
displacy.render(doc, style="dep", jupyter=True, options={"distance": 110})


--- Built-in spaCy Noun Chunks ---
Phrase: The clever machine learning engineer | Root: engineer   | Dependency: nsubj
Phrase: the complex system error            | Root: error      | Dependency: dobj


In [9]:
# Exercise 1: Prepositional Phrase (PP) Chunking
pp_sentence = "The scientist conducted tests in the laboratory during the night."

pp_tokens = word_tokenize(pp_sentence)
pp_tagged = nltk.pos_tag(pp_tokens)

print("--- POS-Tagged Sentence ---")
print(pp_tagged)

pp_grammar = r"""
    PP: {<IN><DT>?<JJ>*<NN.*>+}
"""
pp_parser = nltk.RegexpParser(pp_grammar)
pp_tree = pp_parser.parse(pp_tagged)

print("\n--- Chunk Tree with PP Chunks ---")
print(pp_tree)

print("\n--- Extracted Prepositional Phrases ---")
for subtree in pp_tree.subtrees(filter=lambda t: t.label() == "PP"):
    phrase = " ".join(word for word, tag in subtree.leaves())
    print(f"PP: [{phrase}]")


--- POS-Tagged Sentence ---
[('The', 'DT'), ('scientist', 'NN'), ('conducted', 'VBD'), ('tests', 'NNS'), ('in', 'IN'), ('the', 'DT'), ('laboratory', 'NN'), ('during', 'IN'), ('the', 'DT'), ('night', 'NN'), ('.', '.')]

--- Chunk Tree with PP Chunks ---
(S
  The/DT
  scientist/NN
  conducted/VBD
  tests/NNS
  (PP in/IN the/DT laboratory/NN)
  (PP during/IN the/DT night/NN)
  ./.)

--- Extracted Prepositional Phrases ---
PP: [in the laboratory]
PP: [during the night]


In [10]:
# Exercise 2: Standard Tokenizer vs. TweetTokenizer Comparison
noisy_tweet = "omg @user this is sooo cool!! \U0001F525 https://example.com #nlp"

# 1. Standard tokenizer
standard_tokens = word_tokenize(noisy_tweet)
standard_tags = nltk.pos_tag(standard_tokens)

# 2. TweetTokenizer
tweet_tok = TweetTokenizer(preserve_case=False, strip_handles=False, reduce_len=True)
tweet_tokens = tweet_tok.tokenize(noisy_tweet)
tweet_tags = nltk.pos_tag(tweet_tokens)

max_len = max(len(standard_tags), len(tweet_tags))
rows = []
for i in range(max_len):
    std_tok, std_tag = standard_tags[i] if i < len(standard_tags) else ("", "")
    twt_tok, twt_tag = tweet_tags[i] if i < len(tweet_tags) else ("", "")
    rows.append({
        "Standard_Token": std_tok, "Standard_Tag": std_tag,
        "Tweet_Token": twt_tok, "Tweet_Tag": twt_tag
    })

df_compare = pd.DataFrame(rows)
print("--- Side-by-Side Comparison: word_tokenize vs. TweetTokenizer ---")
print(df_compare.to_string(index=False))

print("""
Observations:
- word_tokenize splits the URL and hashtag into multiple noisy sub-tokens
  (e.g., punctuation, slashes, and the '#' symbol get separated from the text).
- TweetTokenizer preserves '@user', '#nlp', and the URL as single, meaningful tokens,
  and normalizes elongated words like 'sooo' via reduce_len.
- Expressive punctuation ('!!') and emojis are kept intact by TweetTokenizer but may be
  split apart or misclassified by the standard tokenizer.
""")


--- Side-by-Side Comparison: word_tokenize vs. TweetTokenizer ---
Standard_Token Standard_Tag         Tweet_Token Tweet_Tag
           omg           JJ                 omg        RB
             @          NNP               @user       RBR
          user           NN                this        DT
          this           DT                  is       VBZ
            is          VBZ                sooo        JJ
          sooo           JJ                cool        NN
          cool           NN                   !         .
             !            .                   !         .
             !            .                   🔥        JJ
             🔥           JJ https://example.com        NN
         https           NN                #nlp        NN
             :            :                              
 //example.com           JJ                              
             #            #                              
           nlp           NN                              

Obser

In [11]:
# Exercise 3: SVO Triplet Extraction using spaCy Dependency Parsing
def extract_svo_triplets(doc):
    """Extract (Subject, Verb, Object) triplets from a spaCy-parsed document."""
    triplets = []
    for token in doc:
        if token.pos_ == "VERB":
            subject = None
            obj = None
            for child in token.children:
                if child.dep_ in ("nsubj", "nsubjpass"):
                    subject = child.text
                if child.dep_ in ("dobj", "obj"):
                    obj = child.text
            if subject and obj:
                triplets.append((subject, token.text, obj))
    return triplets

test_sentences = [
    "The clever machine learning engineer quickly solved the complex system error.",
    "The scientist conducted tests in the laboratory during the night.",
    "The boy ate an apple."
]

for sent in test_sentences:
    doc_svo = nlp(sent)
    triplets = extract_svo_triplets(doc_svo)
    print(f"Sentence: {sent}")
    print(f"SVO Triplets: {triplets}\n")


Sentence: The clever machine learning engineer quickly solved the complex system error.
SVO Triplets: [('engineer', 'solved', 'error')]

Sentence: The scientist conducted tests in the laboratory during the night.
SVO Triplets: [('scientist', 'conducted', 'tests')]

Sentence: The boy ate an apple.
SVO Triplets: [('boy', 'ate', 'apple')]



In [12]:
# Exercise 4: Chinking Implementation with IOB Tag Extraction
chink_sentence = "The clever machine learning engineer quickly solved the complex system error."
chink_tokens = word_tokenize(chink_sentence)
chink_tagged = nltk.pos_tag(chink_tokens)

exercise_chink_grammar = r"""
    CHUNK:
        {<.*>+}              # Chunk everything
        }<DT|PRP|CC>+{        # Chink out determiners, personal pronouns, and conjunctions
"""

exercise_chink_parser = nltk.RegexpParser(exercise_chink_grammar)
exercise_chink_tree = exercise_chink_parser.parse(chink_tagged)

print("--- Tree after Chinking (DT, PRP, CC removed) ---")
print(exercise_chink_tree)

# Convert to IOB tags
exercise_iob_tags = tree2conlltags(exercise_chink_tree)
df_exercise_chunks = pd.DataFrame(exercise_iob_tags, columns=["Token", "POS_Tag", "IOB_Tag"])

print("\n--- Final Token-Tag-Chunk DataFrame (IOB Format) ---")
print(df_exercise_chunks.to_string(index=False))


--- Tree after Chinking (DT, PRP, CC removed) ---
(S
  The/DT
  (CHUNK
    clever/NN
    machine/NN
    learning/VBG
    engineer/JJ
    quickly/RB
    solved/VBD)
  the/DT
  (CHUNK complex/JJ system/NN error/NN ./.))

--- Final Token-Tag-Chunk DataFrame (IOB Format) ---
   Token POS_Tag IOB_Tag
     The      DT       O
  clever      NN B-CHUNK
 machine      NN I-CHUNK
learning     VBG I-CHUNK
engineer      JJ I-CHUNK
 quickly      RB I-CHUNK
  solved     VBD I-CHUNK
     the      DT       O
 complex      JJ B-CHUNK
  system      NN I-CHUNK
   error      NN I-CHUNK
       .       . I-CHUNK
